# 06A — Frozen-Selector Evaluation on the Final QAOA Holdout

**QuantumOptLab · Milestone 6 · Seed 31 · Qiskit 2.6.2 / Aer 0.18.2**

This is a **one-time, precommitted, graph-disjoint test** of the exact Milestone 5 selector. Its purpose is to test whether the adaptive depth policy generalizes. It does not fit or select a new model.

**Sequence:** (1) validate Phase 4 and Phase 5 artifacts; (2) write and freeze *all* depth decisions using graph/noise inputs alone; (3) optimize QAOA parameters for test graphs using Phase 4's ideal-noise protocol; (4) execute genuine compiled Qiskit circuits in Aer; (5) score paired outcomes and bootstrap **graphs**, not six correlated noise conditions; (6) save a signed-by-hash audit manifest. All numerical results are newly measured on the locked split.

> **Important:** Running in `full` mode reveals the previously untouched 20 test graphs. After this, the test is no longer blind. Report the first full run without redesigning the selector based on its findings. `smoke` uses four *validation* graphs only and **does not unlock the test**.

> This tests a **quantum-classical hybrid selector**, not quantum advantage. The CX gate penalty is a declared engineering proxy, not a real device fee. Parameter optimization happens separately for each evaluation graph using ideal simulation; it is not parameter transfer.

### 1. Configure the one-time evaluation

The default is the full 20-graph test. Set `RUN_MODE = "smoke"` **before the first run** to validate the pipeline using only four development-validation graphs. Keep the seed, shot budget, original noise settings and declared utility weight fixed. The first full-run decisions will be committed to disk before any test objective labels are generated.

In [ ]:
from __future__ import annotations
from pathlib import Path
from hashlib import sha256
import json, platform, time, itertools
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from IPython.display import display
import joblib
import qiskit, qiskit_aer, sklearn
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error

SEED = 31
RUN_MODE = "full"  # change to "smoke" BEFORE running if you do not want to reveal the locked test
assert RUN_MODE in ("full", "smoke")
DEPTHS = (1, 2)
RESTARTS, MAXITER = 2, 55              # frozen Phase 4 optimizer protocol
SHOTS, REPEATS, BATCH_SIZE = 1024, 3, 8
NOISE_CONDITIONS = (
    ("ideal", 0.0, 0.0), ("cx_002", 0.02, 0.0), ("cx_005", 0.05, 0.0),
    ("cx_010", 0.10, 0.0), ("readout_002", 0.0, 0.02),
    ("cx_005_readout_002", 0.05, 0.02),
)
ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks": ROOT = ROOT.parent
assert (ROOT / "notebooks").is_dir(), "Run from QuantumOptLab root or its notebooks directory"
EXP = ROOT / "reports" / "experiments"
FIG = ROOT / "reports" / "figures"
EXP.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
PREFIX = f"phase06_{RUN_MODE}_seed{SEED}"
P4 = f"phase04_full_seed{SEED}"
P5 = f"phase05_full_seed{SEED}"
SOURCES = {
    "p4_catalog": EXP / f"{P4}_graph_catalog.json",
    "p4_ideal": EXP / f"{P4}_qaoa_ideal.csv",
    "p4_validation_noise": EXP / f"{P4}_noise_raw.csv",
    "p4_manifest": EXP / f"{P4}_manifest.json",
    "p5_model": EXP / f"{P5}_frozen_selector.joblib",
    "p5_card": EXP / f"{P5}_frozen_selector_card.json",
    "p5_manifest": EXP / f"{P5}_manifest.json",
    "p5_paired": EXP / f"{P5}_paired_labels.csv",
}
missing = [str(v) for v in SOURCES.values() if not v.is_file()]
if missing: raise FileNotFoundError("Missing full-run Phase 4/5 files:\n" + "\n".join(missing))
SHA = lambda path: sha256(Path(path).read_bytes()).hexdigest()
SOURCE_SHA = {k: SHA(v) for k, v in SOURCES.items()}
START = time.perf_counter()
print("Project:", ROOT)
print("Mode:", RUN_MODE, "| Qiskit:", qiskit.__version__, "| Aer:", qiskit_aer.__version__)
print("Full mode performs the FINAL held-out evaluation. Smoke mode uses validation only.")

### 2. Verify provenance and load the frozen selector

Phase 5's model card must match the exact Phase 4 source hashes. Verify that previous Phase 4 and 5 results never used test labels. The frozen selector's feature order, decision threshold and learned estimator are loaded **without refitting**. Only load `joblib` models created by you in your trusted project folder (pickle-based files can execute code when loaded).

In [ ]:
with SOURCES["p4_catalog"].open(encoding="utf-8") as handle: CATALOG_DOC = json.load(handle)
with SOURCES["p4_manifest"].open(encoding="utf-8") as handle: P4_MANIFEST = json.load(handle)
with SOURCES["p5_card"].open(encoding="utf-8") as handle: CARD = json.load(handle)
with SOURCES["p5_manifest"].open(encoding="utf-8") as handle: P5_MANIFEST = json.load(handle)
assert CARD["run_mode"] == "full" and P5_MANIFEST["run_mode"] == "full"
assert P5_MANIFEST["config_hash"] == CARD["config_hash"]
assert all(P5_MANIFEST["all_checks"].values()), "M5 verification did not complete"
assert P4_MANIFEST["config_hash"] == CATALOG_DOC["config_hash"]
assert CARD["phase4_source_sha256"] == {
    "catalog": SOURCE_SHA["p4_catalog"], "ideal": SOURCE_SHA["p4_ideal"],
    "validation_noise": SOURCE_SHA["p4_validation_noise"], "manifest": SOURCE_SHA["p4_manifest"],
}, "Phase 4 sources changed after Phase 5 freezing"
assert CARD["locked_test_graph_count"] == 20 and CARD["train_graph_count"] == 40
assert abs(float(CARD["lambda_cx"]) - 0.0005) < 1e-14
CATALOG = pd.DataFrame(CATALOG_DOC["records"])
assert len(CATALOG) == 80 and CATALOG.graph_id.is_unique
assert CATALOG.split.value_counts().to_dict() == {"train": 40, "validation": 20, "test": 20}
TEST_IDS = set(CATALOG.loc[CATALOG.split == "test", "graph_id"])
P4_IDEAL = pd.read_csv(SOURCES["p4_ideal"])
P4_NOISE = pd.read_csv(SOURCES["p4_validation_noise"])
P5_PAIRED = pd.read_csv(SOURCES["p5_paired"])
assert set(P4_IDEAL.graph_id).isdisjoint(TEST_IDS)
assert set(P4_NOISE.graph_id).isdisjoint(TEST_IDS)
assert set(P5_PAIRED.graph_id).isdisjoint(TEST_IDS)
FEATURES = list(CARD["features"])
assert len(FEATURES) == 19 and not (set(FEATURES) & {"graph_id", "ratio_p1", "ratio_p2", "optimum", "params_json"})
assert CARD["selected_model_name"] == P5_MANIFEST["selected_model"]
# The model must be loaded from the user's original trusted project, never from an external ZIP.
SELECTOR = joblib.load(SOURCES["p5_model"])
assert hasattr(SELECTOR, "predict") and getattr(SELECTOR, "n_features_in_", len(FEATURES)) == len(FEATURES)
if RUN_MODE == "full":
    EVAL_IDS = sorted(TEST_IDS)
else:
    EVAL_IDS = sorted(CATALOG.loc[CATALOG.split == "validation"].groupby("family", sort=True).head(1).graph_id)
assert len(EVAL_IDS) == (20 if RUN_MODE == "full" else 4)
assert set(EVAL_IDS).issubset(TEST_IDS if RUN_MODE == "full" else set(CATALOG.loc[CATALOG.split == "validation", "graph_id"]))
BY_ID = {r["graph_id"]: r for r in CATALOG_DOC["records"]}
GRAPHS = {}
for gid in EVAL_IDS:
    record = BY_ID[gid]
    graph = nx.Graph()
    graph.add_nodes_from(range(int(record["n"])))
    graph.add_weighted_edges_from((int(u), int(v), int(w)) for u, v, w in record["edge_list"])
    assert nx.is_connected(graph)
    assert nx.weisfeiler_lehman_graph_hash(graph, edge_attr="weight") == record["fingerprint"]
    GRAPHS[gid] = graph
print("Loaded frozen estimator:", CARD["selected_model_name"])
print("Evaluation split:", "LOCKED TEST" if RUN_MODE == "full" else "development smoke", "| graphs:", len(GRAPHS))
print("Phase 4 and Phase 5 provenance verified; selector was not trained here.")

### 3. Commit selector predictions **before** test outcomes

Compute exactly the same allowed graph and noise features used in Phase 5. Write every decision for all six noise conditions, along with a hash of the frozen model and an explicit protocol. Existing commitments must match on a rerun; this blocks silent redefinition of the policy after revealing outcomes.

In [ ]:
def graph_features(graph_id):
    g = GRAPHS[graph_id]
    degrees = np.asarray([degree for _, degree in g.degree()], dtype=float)
    weights = np.asarray([float(d["weight"]) for *_, d in g.edges(data=True)], dtype=float)
    n, edges = g.number_of_nodes(), g.number_of_edges()
    return {
        "graph_id": graph_id, "n": n, "edges": edges,
        "density": nx.density(g), "average_degree": float(degrees.mean()),
        "degree_std": float(degrees.std()), "clustering": float(nx.average_clustering(g)),
        "triangles": float(sum(nx.triangles(g).values()) / 3),
        "weight_sum": float(weights.sum()), "weight_mean": float(weights.mean()),
        "weight_std": float(weights.std()), "max_weight": float(weights.max()),
        "weighted_flag": int(not np.allclose(weights, 1.0)),
    }

D = pd.DataFrame([graph_features(i) for i in EVAL_IDS])
D = D.merge(pd.DataFrame(NOISE_CONDITIONS, columns=["condition", "cx_p", "readout_p"]), how="cross")
D["cx_p1"] = 2 * D.edges
D["cx_p2"] = 4 * D.edges
D["delta_cx"] = D.cx_p2 - D.cx_p1
D["cx_exposure_p1"] = D.cx_p * D.cx_p1
D["cx_exposure_p2"] = D.cx_p * D.cx_p2
D["readout_exposure"] = D.readout_p * D.n
assert D[FEATURES].notna().all().all() and np.isfinite(D[FEATURES].to_numpy()).all()
D["predicted_delta_ratio"] = np.asarray(SELECTOR.predict(D[FEATURES]), dtype=float)
assert np.isfinite(D.predicted_delta_ratio).all()
D["selected_depth"] = np.where(D.predicted_delta_ratio > float(CARD["lambda_cx"]) * D.delta_cx + 1e-12, 2, 1)
D["lookup_depth"] = np.where(D.condition.map(CARD["train_noise_lookup"]) > float(CARD["lambda_cx"]) * D.delta_cx + 1e-12, 2, 1)
D["gate_rule_depth"] = np.where(D.cx_exposure_p1.to_numpy() < float(CARD["gate_threshold"]), 2, 1)
D = D.sort_values(["graph_id", "condition"]).reset_index(drop=True)

PROTOCOL = {
    "project": "QuantumOptLab", "milestone": "6A", "mode": RUN_MODE, "seed": SEED,
    "evaluation_split": "test" if RUN_MODE == "full" else "validation_smoke",
    "model_name": CARD["selected_model_name"], "model_sha256": SOURCE_SHA["p5_model"],
    "model_card_sha256": SOURCE_SHA["p5_card"], "phase5_manifest_sha256": SOURCE_SHA["p5_manifest"],
    "phase4_source_sha256": CARD["phase4_source_sha256"],
    "evaluation_graph_ids": EVAL_IDS, "feature_order": FEATURES,
    "decision_rule": CARD["decision"], "lambda_cx": float(CARD["lambda_cx"]),
    "optimizer": {"method": "L-BFGS-B", "restarts": RESTARTS, "maxiter": MAXITER,
                  "maxfun": 600, "ftol": 1e-9, "target": "ideal_expected_cut",
                  "seed_formula": "31 + 100000*sorted_evaluation_graph_position + 1000*p + restart"},
    "noise_conditions": [list(x) for x in NOISE_CONDITIONS], "shots": SHOTS, "repeats": REPEATS,
    "basis_gates": ["h", "rx", "rz", "cx"], "transpile_level": 0,
    "bootstrap": {"unit": "graph", "draws": 5000, "seed": SEED + 6006},
    "selection_is_frozen": True,
}
PROTO_HASH = sha256(json.dumps(PROTOCOL, sort_keys=True).encode("utf-8")).hexdigest()[:16]
COMMIT_PATH = EXP / f"{PREFIX}_precommitted_decisions.csv"
PROTO_PATH = EXP / f"{PREFIX}_preregistered_protocol.json"
new_csv = D.to_csv(index=False, float_format="%.17g").encode("utf-8")
if COMMIT_PATH.exists():
    assert COMMIT_PATH.read_bytes() == new_csv, "Frozen decision file differs; do NOT change predictions after revealing outcomes"
else:
    COMMIT_PATH.write_bytes(new_csv)
if PROTO_PATH.exists():
    assert json.loads(PROTO_PATH.read_text(encoding="utf-8")) == PROTOCOL, "Frozen experiment protocol changed"
else:
    PROTO_PATH.write_text(json.dumps(PROTOCOL, indent=2), encoding="utf-8")
COMMIT_SHA = SHA(COMMIT_PATH)
assert len(D) == len(EVAL_IDS) * len(NOISE_CONDITIONS)
assert set(D.selected_depth.unique()).issubset({1, 2})
print("PRECOMMITTED:", len(D), "depth decisions; SHA-256:", COMMIT_SHA)
print("Protocol hash:", PROTO_HASH)
display(D.groupby("condition", sort=True).agg(choose_p2_fraction=("selected_depth",lambda x: (x==2).mean()),
                                           mean_predicted_delta=("predicted_delta_ratio", "mean")).round(4))

### 4. Optimize the test QAOA circuits with the **frozen** Phase 4 protocol

Now, and **only now**, compute exact Max-Cut reference costs and optimize circuit angles. The fast amplitude implementation is independently compared with `qiskit.quantum_info.Statevector` before use. For each graph and depth, run two seeded L-BFGS-B restarts to maximize ideal expected cut. New graph parameters are allowed: we are testing **depth selection**, not angle transfer. Optimizer restarts are checkpointed without changing the algorithm or the precommitted decisions.

In [ ]:
def exact_cost_table(g):
    bits = np.arange(1 << g.number_of_nodes(), dtype=np.uint64)
    costs = np.zeros(len(bits), dtype=float)
    for u, v, e in g.edges(data=True):
        costs += float(e["weight"]) * (((bits >> u) ^ (bits >> v)) & 1)
    return costs

COSTS = {gid: exact_cost_table(GRAPHS[gid]) for gid in EVAL_IDS}
OPTIMA = {gid: float(c.max()) for gid, c in COSTS.items()}
assert all(v > 0 for v in OPTIMA.values())

def qaoa_circuit(g, params, p):
    params = np.asarray(params, dtype=float)
    assert params.shape == (2 * p,)
    qc = QuantumCircuit(g.number_of_nodes())
    qc.h(range(g.number_of_nodes()))
    for gamma, beta in zip(params[:p], params[p:]):
        for u, v, data in g.edges(data=True):
            qc.rzz(-float(gamma) * float(data["weight"]), u, v)
        for qubit in range(g.number_of_nodes()):
            qc.rx(2 * float(beta), qubit)
    return qc

def fast_probs(n, costs, params, p):
    a = np.full(1 << n, 1 / np.sqrt(1 << n), dtype=np.complex128)
    bits = np.arange(1 << n, dtype=np.uint64)
    pairs = []
    for i in range(n):
        indices = np.flatnonzero(((bits >> i) & 1) == 0)
        pairs.append((indices, indices | (1 << i)))
    for gamma, beta in zip(np.asarray(params)[:p], np.asarray(params)[p:]):
        a *= np.exp(-1j * float(gamma) * costs)
        c, s = np.cos(beta), -1j * np.sin(beta)
        for left_ids, right_ids in pairs:
            l, r = a[left_ids].copy(), a[right_ids].copy()
            a[left_ids] = c*l + s*r
            a[right_ids] = s*l + c*r
    return np.abs(a)**2

rng = np.random.default_rng(SEED + 606)
physics_errors = []
for gid in [EVAL_IDS[0], EVAL_IDS[-1]]:
    for p in DEPTHS:
        pars = np.r_[rng.uniform(0,2*np.pi,p), rng.uniform(0,np.pi,p)]
        exact = np.abs(Statevector.from_instruction(qaoa_circuit(GRAPHS[gid], pars, p)).data)**2
        fast = fast_probs(GRAPHS[gid].number_of_nodes(), COSTS[gid], pars, p)
        physics_errors.append(np.abs(exact-fast).max())
assert max(physics_errors) < 1e-10
print("Qiskit Statevector cross-check max error:", f"{max(physics_errors):.3e}")

TRIAL_PATH = EXP / f"{PREFIX}_optimizer_restarts.csv"
if TRIAL_PATH.exists():
    trial_rows = pd.read_csv(TRIAL_PATH).to_dict("records")
    assert all(row["protocol_hash"] == PROTO_HASH for row in trial_rows)
else: trial_rows = []
seen = {(r["graph_id"], int(r["depth_p"]), int(r["restart"])) for r in trial_rows}
assert len(seen) == len(trial_rows), "Duplicate optimization checkpoint rows"
for position, gid in enumerate(EVAL_IDS):
    for p in DEPTHS:
        for restart in range(RESTARTS):
            if (gid,p,restart) in seen: continue
            opt_seed = SEED + 100_000 * position + 1000*p + restart
            gen = np.random.default_rng(opt_seed)
            initial = np.r_[gen.uniform(0,2*np.pi,p), gen.uniform(0,np.pi,p)]
            cost = COSTS[gid]; n = GRAPHS[gid].number_of_nodes()
            f = lambda par: -float(fast_probs(n, cost, par, p) @ cost)
            result = minimize(f, initial, method="L-BFGS-B",
                              bounds=[(0,2*np.pi)]*p + [(0,np.pi)]*p,
                              options={"maxiter": MAXITER, "maxfun":600, "ftol":1e-9})
            probs = fast_probs(n, cost, result.x, p)
            val = float(probs @ cost)
            trial_rows.append({"protocol_hash": PROTO_HASH, "graph_id":gid, "depth_p":p,
                "restart":restart, "optimizer_seed":opt_seed, "expected_cut":val,
                "approx_ratio":val/OPTIMA[gid], "p_optimal":float(probs[np.isclose(cost, OPTIMA[gid])].sum()),
                "params_json":json.dumps([float(x) for x in result.x]),
                "nfev":int(result.nfev), "nit":int(result.nit), "converged":bool(result.success)})
            pd.DataFrame(trial_rows).to_csv(TRIAL_PATH, index=False)
            seen.add((gid,p,restart))
    if (position+1)%5==0: print("Optimized", position+1, "of", len(EVAL_IDS), "graphs")
trials = pd.DataFrame(trial_rows)
assert len(trials) == len(EVAL_IDS) * 2 * RESTARTS
BEST = trials.sort_values("expected_cut", ascending=False).drop_duplicates(["graph_id","depth_p"])
BEST = BEST.sort_values(["graph_id","depth_p"]).reset_index(drop=True)
assert BEST.groupby("graph_id").size().eq(2).all()
BEST.to_csv(EXP / f"{PREFIX}_qaoa_ideal.csv", index=False)
print("Saved",len(trials),"optimization restarts,",len(BEST),"best circuits")
display(BEST.groupby("depth_p")[["approx_ratio","p_optimal"]].mean().round(4))

### 5. Compile real Qiskit circuits into CX gates

Build Qiskit `QuantumCircuit` objects using the newly optimized QAOA angles. `transpile(..., optimization_level=0)` preserves the Phase 4 CX-based decomposition, then `measure_all()` supplies the bitstrings used to evaluate Max-Cut. We save actual compiled resources rather than inferring all of them from a graph formula.

In [ ]:
COMPILED = {}
resource_rows = []
for gid in EVAL_IDS:
    g = GRAPHS[gid]
    for p in DEPTHS:
        row = BEST.loc[(BEST.graph_id == gid) & (BEST.depth_p == p)].iloc[0]
        qc = qaoa_circuit(g, json.loads(row.params_json), p)
        compiled = transpile(qc, basis_gates=["h","rx","rz","cx"], optimization_level=0, seed_transpiler=SEED)
        cx = int(compiled.count_ops().get("cx", 0))
        assert cx == 2 * g.number_of_edges() * p
        assert "rzz" not in compiled.count_ops()
        compiled.measure_all()
        COMPILED[gid,p] = compiled
        resource_rows.append({"graph_id":gid,"depth_p":p,"cx_gates":cx,
                              "compiled_depth":int(compiled.depth()),
                              "operations":int(sum(compiled.count_ops().values()))})
RESOURCES = pd.DataFrame(resource_rows)
RESOURCES.to_csv(EXP / f"{PREFIX}_circuit_resources.csv", index=False)
assert len(COMPILED) == 2*len(EVAL_IDS)
print("Qiskit compiled CX-based circuits:",len(COMPILED))
display(RESOURCES.groupby("depth_p")[['cx_gates','compiled_depth']].mean().round(2))
print("Illustrative compiled QAOA circuit (text; first graph, depth 1):")
print(COMPILED[EVAL_IDS[0],1].draw(output="text", fold=75))

### 6. Execute Aer under the **predeclared** noise conditions

For each graph, depth and noise setting, run three deterministic 1,024-shot `AerSimulator(method="density_matrix")` repetitions. The simulator adds depolarizing errors to each compiled CX and independently adds symmetric readout errors. Complete **fixed batches** are replayed after interruptions to preserve RNG ordering. This is simulated noise, not an IBM QPU calibration.

In [ ]:
def make_noise(cx_p, readout_p):
    noise = NoiseModel()
    if cx_p: noise.add_all_qubit_quantum_error(depolarizing_error(float(cx_p), 2), ["cx"])
    if readout_p:
        r = float(readout_p)
        noise.add_all_qubit_readout_error(ReadoutError([[1-r,r],[r,1-r]]))
    return noise

def summarize_counts(gid, counts):
    cost, optimum = COSTS[gid], OPTIMA[gid]
    count_total = sum(int(n) for n in counts.values())
    score_sum = 0.0; n_opt = 0
    for bits, count in counts.items():
        assignment = int(bits.replace(" ", ""), 2)
        score_sum += float(cost[assignment]) * int(count)
        if np.isclose(cost[assignment], optimum): n_opt += int(count)
    return float(score_sum/count_total), float(n_opt/count_total), count_total

RAW_PATH = EXP / f"{PREFIX}_noise_raw.csv"
if RAW_PATH.exists():
    samples = pd.read_csv(RAW_PATH).to_dict("records")
    assert all(row["protocol_hash"] == PROTO_HASH for row in samples)
else: samples = []
completed = {(r["graph_id"],int(r["depth_p"]),r["condition"],int(r["repeat"])) for r in samples}
assert len(completed) == len(samples), "Duplicate sample checkpoint entries"
ordered = list(itertools.product(EVAL_IDS,DEPTHS))
for condition_idx,(condition,cx_p,readout_p) in enumerate(NOISE_CONDITIONS):
    sim = AerSimulator(method="density_matrix", noise_model=make_noise(cx_p,readout_p), max_parallel_experiments=1)
    for repeat in range(REPEATS):
        for offset in range(0,len(ordered),BATCH_SIZE):
            batch = ordered[offset:offset+BATCH_SIZE]
            if all((gid,p,condition,repeat) in completed for gid,p in batch): continue
            simulator_seed = SEED + 3_000_000 + condition_idx * 100_000 + repeat * 1000 + offset // BATCH_SIZE
            result = sim.run([COMPILED[gid,p] for gid,p in batch], shots=SHOTS,
                             seed_simulator=simulator_seed).result()
            assert result.success, "Aer simulation failed"
            for idx,(gid,p) in enumerate(batch):
                key = (gid,p,condition,repeat)
                if key in completed: continue
                value,popt,count = summarize_counts(gid,result.get_counts(idx))
                assert count == SHOTS
                samples.append({"protocol_hash":PROTO_HASH,"graph_id":gid,"depth_p":p,
                    "condition":condition,"cx_p":cx_p,"readout_p":readout_p,
                    "repeat":repeat,"simulator_seed":simulator_seed,"shots":count,
                    "expected_cut_shots":value,"approx_ratio":value/OPTIMA[gid],"p_optimal":popt})
                completed.add(key)
            pd.DataFrame(samples).to_csv(RAW_PATH,index=False)
    print("Aer:", condition, "finished")
RAW = pd.DataFrame(samples)
expected = len(EVAL_IDS)*2*len(NOISE_CONDITIONS)*REPEATS
assert len(RAW) == expected and not RAW.duplicated(["graph_id","depth_p","condition","repeat"]).any()
assert RAW.shots.eq(SHOTS).all() and np.isfinite(RAW[["approx_ratio","p_optimal"]].to_numpy()).all()
print("Completed",len(RAW),"new Aer executions |",len(EVAL_IDS),"evaluation graphs")

### 7. Score frozen policies and paired graph-level uncertainty

Only after the precommitted decisions and simulation outcomes are saved, associate a depth choice with its measured reward. Compare the selected model, always p=1, always p=2, the frozen Phase 5 lookup and gate rule, and an **oracle upper bound** that has access to both outcomes (not deployable). Confidence intervals resample whole graphs, keeping their six conditions together. Compare approximation ratio and optimal-solution probability **separately** from the declared CX-penalized utility.

In [ ]:
SUMMARY_RAW = RAW.groupby(["graph_id","depth_p","condition"], as_index=False).agg(
    ratio=("approx_ratio","mean"), optimal_probability=("p_optimal","mean"))
R = SUMMARY_RAW.pivot(index=["graph_id","condition"], columns="depth_p", values=["ratio","optimal_probability"])
R.columns = [f"{key}_p{depth}" for key,depth in R.columns]
R = R.reset_index()
TABLE = D.merge(R,on=["graph_id","condition"],validate="one_to_one")
assert len(TABLE) == len(D) and TABLE[["ratio_p1","ratio_p2"]].notna().all().all()
TABLE["utility_p1"] = TABLE.ratio_p1 - float(CARD["lambda_cx"]) * TABLE.cx_p1
TABLE["utility_p2"] = TABLE.ratio_p2 - float(CARD["lambda_cx"]) * TABLE.cx_p2
TABLE["oracle_depth"] = np.where(TABLE.utility_p2 > TABLE.utility_p1 + 1e-12,2,1)
POLICIES = {
    "selected_cv_model": TABLE.selected_depth.to_numpy(dtype=int),
    "always_p1": np.ones(len(TABLE),dtype=int),
    "always_p2": np.full(len(TABLE),2,dtype=int),
    "train_noise_lookup":TABLE.lookup_depth.to_numpy(dtype=int),
    "gate_budget_rule":TABLE.gate_rule_depth.to_numpy(dtype=int),
    "oracle_upper_bound":TABLE.oracle_depth.to_numpy(dtype=int),
}
records = []
for policy, choices in POLICIES.items():
    ratio = np.where(choices == 2,TABLE.ratio_p2,TABLE.ratio_p1)
    popt = np.where(choices == 2,TABLE.optimal_probability_p2,TABLE.optimal_probability_p1)
    cx = np.where(choices == 2,TABLE.cx_p2,TABLE.cx_p1)
    utility = ratio-float(CARD["lambda_cx"])*cx
    regret = np.maximum(TABLE.utility_p1,TABLE.utility_p2)-utility
    assert (regret >= -1e-10).all()
    for i in range(len(TABLE)):
        records.append({"policy":policy,"graph_id":TABLE.graph_id.iloc[i],
             "family":BY_ID[TABLE.graph_id.iloc[i]]["family"],
             "condition":TABLE.condition.iloc[i],"depth_p":int(choices[i]),
             "oracle_depth":int(TABLE.oracle_depth.iloc[i]),
             "approx_ratio":float(ratio[i]),"p_optimal":float(popt[i]),
             "utility":float(utility[i]),"regret":float(regret[i]),"cx_gates":int(cx[i])})
POLICY = pd.DataFrame(records)
POLICY_PATH = EXP / f"{PREFIX}_policy_decisions.csv"
POLICY.to_csv(POLICY_PATH,index=False)
AGG = POLICY.groupby("policy",as_index=False).agg(
    n_decisions=("utility","size"),mean_utility=("utility","mean"),
    mean_ratio=("approx_ratio","mean"),mean_optimal_probability=("p_optimal","mean"),
    mean_regret=("regret","mean"),mean_cx_gates=("cx_gates","mean"),
    choose_p2_fraction=("depth_p",lambda v: float((v==2).mean())),
)
AGG = AGG.sort_values("mean_utility",ascending=False).reset_index(drop=True)
AGG.to_csv(EXP / f"{PREFIX}_policy_summary.csv",index=False)
print("Frozen policy results on:","LOCKED TEST" if RUN_MODE=="full" else "development smoke")
display(AGG.round(5))

G = POLICY.pivot_table(index=["graph_id"],columns="policy",values="utility",aggfunc="mean")
selected = G["selected_cv_model"].to_numpy(dtype=float)
rng = np.random.default_rng(SEED+6006)
BOOT_INDICES = rng.integers(0,len(G),size=(5000,len(G)))
ci_rows = []
for competitor in ["always_p1","always_p2","train_noise_lookup","gate_budget_rule"]:
    diff = selected - G[competitor].to_numpy(dtype=float)
    samples_boot = diff[BOOT_INDICES].mean(axis=1)
    lo,hi = np.quantile(samples_boot,[.025,.975])
    ci_rows.append({"selected_minus":competitor,"n_graphs":len(G),
                    "utility_gain":float(diff.mean()),"ci95_low":float(lo),"ci95_high":float(hi)})
CI = pd.DataFrame(ci_rows)
CI.to_csv(EXP / f"{PREFIX}_graph_bootstrap_comparisons.csv",index=False)
print("Paired 95% graph-bootstrap intervals for selector gains:")
display(CI.round(5))

### 8. Save figures and audit the integrity of the one-time result

Plot the measured quality/circuit-cost tradeoff and the frozen selector's behavior across noise conditions. Keep axis labels readable for project screenshots. The final checks verify source provenance, untouched model bytes, decision-first sequencing, complete paired Aer measurements, CX circuit compilation, and arithmetic consistency. The manifest records the observed results without changing the frozen protocol.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10.7,3.8))
show = ["always_p1","always_p2","train_noise_lookup","gate_budget_rule","selected_cv_model"]
plot = AGG.set_index("policy").loc[show]
xx = np.arange(len(plot))
axes[0].bar(xx,plot.mean_ratio)
axes[0].set_xticks(xx,["p=1","p=2","Lookup","Gate rule","Adaptive"],rotation=18,ha="right")
axes[0].set_ylim(0,1.05); axes[0].set_title("Evaluation: Mean Max-Cut Approximation Ratio")
axes[0].set_ylabel("Measured approximation ratio (higher is better)")
axes[0].grid(axis="y",alpha=0.2)
axes[1].bar(xx,plot.mean_cx_gates)
axes[1].set_xticks(xx,["p=1","p=2","Lookup","Gate rule","Adaptive"],rotation=18,ha="right")
axes[1].set_title("Quantum Circuit CX Operations")
axes[1].set_ylabel("Mean compiled CX count (lower is better)")
axes[1].grid(axis="y",alpha=0.2)
fig.tight_layout()
FIG_A = FIG / f"{PREFIX}_quality_vs_cost.png"
fig.savefig(FIG_A,dpi=175,bbox_inches="tight");plt.show()

fig,ax = plt.subplots(figsize=(8.5,3.7))
by_condition = POLICY[POLICY.policy=="selected_cv_model"].groupby("condition").agg(
    choose_p2=("depth_p",lambda x:(x==2).mean()),
    mean_regret=("regret","mean"))
order=[c[0] for c in NOISE_CONDITIONS]
ax.bar(np.arange(len(order)),by_condition.reindex(order).choose_p2)
ax.set_xticks(np.arange(len(order)),order,rotation=28,ha="right")
ax.set_ylabel("Fraction selecting depth 2")
ax.set_ylim(0,1.05)
ax.set_title("Frozen Adaptive Depth Decisions Across Noise Conditions")
ax.grid(axis="y",alpha=0.2)
fig.tight_layout()
FIG_B=FIG/f"{PREFIX}_depth_by_condition.png"
fig.savefig(FIG_B,dpi=175,bbox_inches="tight");plt.show()

CHECKS = {
    "Original Phase 4 provenance matches M5 frozen model card": CARD["phase4_source_sha256"] == {
        "catalog":SOURCE_SHA["p4_catalog"],"ideal":SOURCE_SHA["p4_ideal"],
        "validation_noise":SOURCE_SHA["p4_validation_noise"],"manifest":SOURCE_SHA["p4_manifest"]},
    "Phase 5 model and manifest not modified": SOURCE_SHA["p5_model"]==SHA(SOURCES["p5_model"]) and SOURCE_SHA["p5_manifest"]==SHA(SOURCES["p5_manifest"]),
    "Every selected graph belongs exclusively to chosen split": set(EVAL_IDS).issubset(TEST_IDS if RUN_MODE=="full" else set(CATALOG.loc[CATALOG.split=="validation","graph_id"])),
    "Only frozen graph and noise features used": FEATURES==CARD["features"] and "ratio_p1" not in FEATURES,
    "Decisions were committed before generating test labels": COMMIT_PATH.exists() and PROTO_PATH.exists() and SHA(COMMIT_PATH)==COMMIT_SHA,
    "Predeclared protocol unmodified": json.loads(PROTO_PATH.read_text(encoding="utf-8"))==PROTOCOL,
    "All 2-depth predictions exist": len(D)==len(EVAL_IDS)*len(NOISE_CONDITIONS) and set(D.selected_depth.unique()).issubset({1,2}),
    "Correct seeded ideal QAOA optimizer restarts": len(trials)==len(EVAL_IDS)*2*RESTARTS and len(BEST)==len(EVAL_IDS)*2,
    "Fast mathematical QAOA matches Qiskit Statevector": max(physics_errors)<1e-10,
    "Compiled CX gates match frozen Phase 4 convention": all(r.cx_gates==2*GRAPHS[r.graph_id].number_of_edges()*r.depth_p for r in RESOURCES.itertuples()),
    "All new Aer measurements complete exactly once": len(RAW)==expected and not RAW.duplicated(["graph_id","depth_p","condition","repeat"]).any(),
    "Measured approximations physically bounded": RAW.approx_ratio.between(0,1+1e-10).all() and RAW.p_optimal.between(0,1).all(),
    "All depth pairs have complete repeated measurements": RAW.groupby(["graph_id","condition","depth_p"]).size().eq(REPEATS).all(),
    "Every policy evaluated on identical graph/condition rows": POLICY.groupby("policy").size().eq(len(D)).all(),
    "Oracle regret nonnegative and zero for oracle": POLICY.regret.ge(-1e-10).all() and POLICY.loc[POLICY.policy=="oracle_upper_bound","regret"].abs().max()<1e-10,
    "Frozen model rewards and circuit cost consistent": np.allclose(POLICY.utility,POLICY.approx_ratio-float(CARD["lambda_cx"])*POLICY.cx_gates),
    "Bootstrap samples whole evaluation graphs": len(G)==len(EVAL_IDS) and len(CI)==4,
    "New results and figures saved": all(x.exists() for x in [RAW_PATH, POLICY_PATH, FIG_A, FIG_B]),
}
CHECKS = {name: bool(value) for name, value in CHECKS.items()}
for name,good in CHECKS.items():print(("PASS" if good else "FAIL")+" | "+name)
assert all(CHECKS.values()), "M6 integrity failure: do NOT publish until resolved"
manifest={
    "project":"QuantumOptLab","phase":"6A","run_mode":RUN_MODE,
    "final_holdout_unlocked":RUN_MODE=="full","first_full_run_is_only_blind_run":True,
    "protocol_hash":PROTO_HASH,"protocol_file":str(PROTO_PATH.relative_to(ROOT)),
    "precommitted_decisions_sha256":COMMIT_SHA,"source_sha256":SOURCE_SHA,
    "qiskit_version":qiskit.__version__,"qiskit_aer_version":qiskit_aer.__version__,
    "sklearn_version":sklearn.__version__,"python_version":platform.python_version(),
    "n_evaluation_graphs":len(EVAL_IDS),"n_new_aer_executions":len(RAW),
    "selected_mean_utility":float(AGG.set_index("policy").loc["selected_cv_model","mean_utility"]),
    "selected_mean_ratio":float(AGG.set_index("policy").loc["selected_cv_model","mean_ratio"]),
    "selected_mean_cx":float(AGG.set_index("policy").loc["selected_cv_model","mean_cx_gates"]),
    "ci95_selected_vs_baselines": CI.to_dict("records"),
    "checks":CHECKS,"elapsed_seconds":round(time.perf_counter()-START,2),
    "limitations":[
        "The first full run unlocks the previously unseen 20 graph test set; later re-runs are not blind.",
        "QAOA angles are optimized separately on each test graph under ideal simulation, not transferred.",
        "All noise settings are synthetic and do not model the full error characteristics of a real quantum chip.",
        "The lambda CX penalty represents declared preference, not measured physical cost.",
        "Graph-family holdout is not tested; test shares families and size strata with training.",
        "No quantum speedup against classical Max-Cut algorithms is demonstrated.",
    ],
}
MANIFEST_PATH=EXP/f"{PREFIX}_manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
print(f"\nPASS: {sum(CHECKS.values())}/{len(CHECKS)} Phase 6A integrity checks")
print("Frozen selector:",CARD["selected_model_name"],"| audit:",MANIFEST_PATH)
if RUN_MODE=="full":print("LOCKED TEST UNLOCKED. Preserve this run; do not retune and present a second test as blind.")
else:print("SMOKE ONLY. Locked test remains untouched.")